# CellMatchR – TabPFN

This notebook runs the CellMatchR TabPFN pipeline for cell type classification of kidney transcriptomic data.

Run the cells below in order using the play icons next to them. All Downloads and dependencies will stay in the Google colab virtual machine.

In [ ]:
#@title 1. Download CellMatchR scripts from GitHub
#@markdown This pulls the TabPFN_scripts subfolder with the relevant scripts from the [CellMatchR repository](https://github.com/genepi-freiburg/CellMatchR) to the current virtual machine.

import os

# Clean up any previous run
!rm -rf CellMatchR

# Clone only the TabPFN_scripts subfolder using sparse checkout
!git clone --filter=blob:none --sparse https://github.com/genepi-freiburg/CellMatchR.git

%cd CellMatchR
!git sparse-checkout set TabPFN_scripts
%cd TabPFN_scripts


In [ ]:
#@title 2. Install Dependencies


print("Installing dependencies, please wait...")
# Install in quiet mode:
!pip install -r requirements.txt -q --no-warn-conflicts

print("✅ Dependencies installed successfully")

In [ ]:
#@title 3. Log in to HuggingFace
#@markdown TabPFN is a gated model.
#@markdown 1. Create an account at [huggingface.co](https://huggingface.co/join)
#@markdown 2. Request access to [TabPFN](https://docs.priorlabs.ai/how-to-access-gated-models)
#@markdown 3. Paste your token from [settings/tokens](https://huggingface.co/settings/tokens) below.

from huggingface_hub import login
try:
    login()
    print("✅ Logged in")
except Exception as e:
    print(f"Login failed or interrupted: {e}")

### 4. Data Settings:
>  **Security & Data Privacy**
>
> **Attention:** This notebook runs on a Virtual Machine (VM) hosted by Google.
>
> * **Temporary Data Storage:** All uploaded files and generated results are stored in the VM's volatile session storage. This data is **automatically and permanently deleted** as soon as the runtime is disconnected or the session times out.
> * **Data Permissions & Responsibility:** You are solely responsible for the data you upload. Please ensure you have the necessary **permissions, licenses, or ethical approvals** before uploading any sensitive or proprietary datasets.
> * **Privacy Policy:** For more details, please refer to the [Google Colab FAQ](https://research.google.com/colaboratory/faq.html#privacy) and the [Google Privacy Policy](https://policies.google.com/privacy).
>



### **Your Dataset**
Upload your CSV (one row per cell, one column per gene symbols). Optionally, add a `meta_target` column with a cell type label per cell to your CSV. If present, an accuracy is calculated and the labels are used as titles in the output probability plots. This is useful for verifying predictions against expected annotations.


<table><tr><td>

**Option 1** without labels

| Slc12a1 | Umod | Nphs1 | Cdh1 | Lrp2 |
|---------|------|-------|------|------|
| 0 | 0 | 0 | 524 | 1203 |
| 0 | 312 | 0 | 0 | 0 |
| 87 | 0 | 0 | 0 | 0 |
| 0 | 0 | 445 | 0 | 0 |

</td><td>&nbsp;&nbsp;&nbsp;&nbsp;</td><td>

**Option 2** with `meta_target` label column

| meta_target | Slc12a1 | Umod | Nphs1 | Cdh1 | Lrp2 |
|-------------|---------|------|-------|------|------|
| PT | 0 | 0 | 0 | 524 | 1203 |
| LOH | 0 | 312 | 0 | 0 | 0 |
| LOH | 87 | 0 | 0 | 0 | 0 |
| POD | 0 | 0 | 445 | 0 | 0 |

</td></tr></table>


> **Note:** Our model is trained on the following cell types: `CD`, `CNT`, `DCT`, `EC`, `ENDO`, `FIB`, `IMM`, `LOH`, `POD`, `PT`. If your `meta_target` column contains other cell type labels, the reported accuracy will not be meaningful, but the labels will still be used for visualization purposes.

### **Reference Settings**
 By default, the model uses our pre-integrated kidney atlas. You can select below if you just want to use a subset of references.

In [ ]:
#@title Upload Data & Configure Settings

#@markdown ### Reference Datasets
#@markdown Select which kidney atlas datasets to use as training reference.
#@markdown Leave all unchecked to use **all available** reference data.
KPMP = False      #@param {type:"boolean"}
Park = False      #@param {type:"boolean"}
Ransick = False   #@param {type:"boolean"}
Zhang = False     #@param {type:"boolean"}

_selected = []

if KPMP:    _selected.append("KPMP")
if Park:    _selected.append("Park")
if Ransick: _selected.append("Ransick")
if Zhang:   _selected.append("Zhang")

REFERENCE_DATASETS = _selected if _selected else None
print(f"Reference datasets: {REFERENCE_DATASETS or 'all available'}")

#@markdown _For more info about the source and preprocessing of the reference datasets see our manuscript and supplementary material._



#@markdown ### Your Data (optional)
#@markdown Check the box below to upload a CSV/TSV file.
#@markdown Leave unchecked to run the demo with the Chen dataset.
UPLOAD_FILE = False  #@param {type:"boolean"}

CSV_PATH = None
if UPLOAD_FILE:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        CSV_PATH = list(uploaded.keys())[0]
        print(f"✓ Uploaded: {CSV_PATH}")
    else:
        print("⚠️ Upload cancelled — falling back to demo data.")
else:
    print("No upload — will use demo data (Chen dataset).")

In [ ]:
#@title 5. Run CellMatchR Pipeline
#@markdown This cell performs data loading, preprocessing, and model prediction.

import os
import logging
import pandas as pd
from IPython.display import display, Markdown
from utils import load_data, prepare_X_y
from main import fit_predict_evaluate
from config import KNOWN_CELL_TYPES

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

if CSV_PATH is not None:
    test_settings = load_data(csv_path=CSV_PATH, reference_datasets=REFERENCE_DATASETS)
else:
    test_settings = load_data(test_datasets=["Chen"])

os.makedirs("results", exist_ok=True)
all_results = {}

for name, (reference_data, test_data) in test_settings.items():
    logger.info(f"=== {name} ===")
    X_train, y_train, X_test, y_test = prepare_X_y(reference_data, test_data)
    logger.info(f"Reference: {X_train.shape[0]} cells, {X_train.shape[1]} genes")
    logger.info(f"Query:     {X_test.shape[0]} cells, {X_test.shape[1]} genes")

    results = fit_predict_evaluate(X_train, y_train, X_test, y_test)

    results["y_test"] = y_test

    has_labels = y_test is not None
    if has_labels:
        unknown_labels = set(y_test.unique()) - KNOWN_CELL_TYPES
        if unknown_labels:
            print(f"⚠️ '{name}' contains labels not in training set.")
            results["acc"] = None
        else:
            print(f"✅ Accuracy: {results['acc']:.1%}")

    summary = results["probs"].copy()
    summary.insert(0, "predicted", results["preds"])
    if has_labels: summary.insert(0, "true_label", y_test.values)

    display(Markdown(f"### {name}"))
    display(summary.style.background_gradient(cmap="Blues", subset=results["probs"].columns))

    out_df = results["probs"].copy()
    out_df.insert(0, "predicted", results["preds"])
    if has_labels: out_df.insert(0, "meta_target", y_test.values)
    out_df.to_csv(f"results/results_{name}.csv", index=False)
    all_results[name] = results

In [ ]:
#@title plotting helpers

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

dataset_names = list(all_results.keys())
state = {"idx": 0}

dropdown = widgets.Dropdown(options=dataset_names, value=dataset_names[0], description="Dataset:")
btn_prev = widgets.Button(description="← Prev")
btn_next = widgets.Button(description="Next →")
label = widgets.HTML()
out = widgets.Output()

def render():
    r = all_results[dropdown.value]
    probs, y_test = r["probs"], r["y_test"]
    n = len(probs)
    state["idx"] = state["idx"] % n

    row = probs.iloc[state["idx"]]
    predicted = row.idxmax()
    confidence = row.max()

    true_str = ""
    if y_test is not None:
        true_label = y_test.iloc[state["idx"]]
        true_str = f" &nbsp;|&nbsp; True: <b>{true_label}</b>"

    label.value = (
        f"<b>Cell {state['idx']+1} / {n}</b> &nbsp;|&nbsp; "
        f"Predicted: <b>{predicted}</b> ({confidence:.1%}){true_str}"
    )

    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        colors = ["#2563eb" if c == predicted else "#cbd5e1" for c in row.index]
        ax.bar(row.index, row.values, color=colors)
        ax.set_ylabel("Probability")
        ax.set_ylim(0, 1)

        title = f"{dropdown.value} "
        if y_test is not None:
            title += f"  Given target: {y_test.iloc[state['idx']]}"
        ax.set_title(title)

        plt.xticks(rotation=45, ha="right")
        fig.tight_layout()
        plt.show()

def on_prev(_):
    n = len(all_results[dropdown.value]["probs"])
    state["idx"] = (state["idx"] - 1) % n
    render()

def on_next(_):
    n = len(all_results[dropdown.value]["probs"])
    state["idx"] = (state["idx"] + 1) % n
    render()

def on_change(_):
    state["idx"] = 0
    render()

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
dropdown.observe(on_change, names="value")

render()
display(widgets.VBox([dropdown, widgets.HBox([btn_prev, btn_next]), label, out]))

## 6. Download results


In [ ]:
#@title Download results
#@markdown run this cell to zip the results and download them

import shutil

# Save static plots
for name, r in all_results.items():
    probs, y_test = r["probs"], r["y_test"]
    n_plots = (len(probs) + 8) // 9
    for pi in range(n_plots):
        batch = probs.iloc[pi*9:(pi+1)*9]
        fig, axes = plt.subplots(3, 3, figsize=(14, 10))
        for ax, (idx, row) in zip(axes.flat, batch.iterrows()):
            colors = ["#2563eb" if c == row.idxmax() else "#cbd5e1" for c in row.index]
            ax.bar(row.index, row.values, color=colors)
            ax.set_ylabel("Probability")
            ax.set_ylim(0, 1)
            title = f"Sample {idx}" if y_test is None else str(y_test.loc[idx])
            ax.set_title(title)
            ax.tick_params(axis="x", rotation=45)
        for ax in axes.flat[len(batch):]:
            ax.set_visible(False)
        fig.tight_layout()
        fig.savefig(f"results/probabilities_{name}_part{pi+1}.png", dpi=150)
        plt.close(fig)

# Zip and download
shutil.make_archive("/content/cellmatchr_results", "zip", "results")

from google.colab import files
files.download("/content/cellmatchr_results.zip")